# 04 - Feature Engineering (Daily Frequency)

**Outputs**:
- `daily_features.parquet` -- daily features for all tickers
- `feature_summary.csv` -- feature descriptions and statistics

**Universe**: 10 tickers (AAPL, AMZN, AVGO, GOOG, GOOGL, META, MSFT, NVDA, TSLA, WMT)
**Frequency**: Daily (all trading days, not monthly snapshots)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROCESSED = Path('../data/processed')
OUTPUT_DIR = PROCESSED

UNIVERSE = ['AAPL', 'AMZN', 'AVGO', 'GOOG', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA', 'WMT']

print(f"Universe: {UNIVERSE}")

Universe: ['AAPL', 'AMZN', 'AVGO', 'GOOG', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA', 'WMT']


## 1. Load Cleaned Data

In [2]:
daily = pd.read_parquet(PROCESSED / 'daily_clean.parquet')
income = pd.read_parquet(PROCESSED / 'income_clean.parquet')
balance = pd.read_parquet(PROCESSED / 'balance_clean.parquet')
cashflow = pd.read_parquet(PROCESSED / 'cashflow_clean.parquet')
overview = pd.read_parquet(PROCESSED / 'overview_clean.parquet')

daily = daily[daily['symbol'].isin(UNIVERSE)].copy()
income = income[income['symbol'].isin(UNIVERSE)].copy()
balance = balance[balance['symbol'].isin(UNIVERSE)].copy()
cashflow = cashflow[cashflow['symbol'].isin(UNIVERSE)].copy()
overview = overview[overview['symbol'].isin(UNIVERSE)].copy()

daily['date'] = pd.to_datetime(daily['date'])
income['fiscalDateEnding'] = pd.to_datetime(income['fiscalDateEnding'])
balance['fiscalDateEnding'] = pd.to_datetime(balance['fiscalDateEnding'])
cashflow['fiscalDateEnding'] = pd.to_datetime(cashflow['fiscalDateEnding'])

daily = daily.sort_values(['symbol', 'date']).reset_index(drop=True)

print(f"Daily prices: {daily.shape}")
print(f"Date range: {daily['date'].min().date()} to {daily['date'].max().date()}")

Daily prices: (52486, 10)
Date range: 2000-01-03 to 2025-12-31


## 2. Technical Features (Daily)

In [3]:
def compute_technical_features(df):
    df = df.copy().sort_values('date')

    # Returns
    df['daily_return'] = df['adjusted_close'].pct_change()

    # Realized volatility (annualized)
    df['vol_5d'] = df['daily_return'].rolling(5).std() * np.sqrt(252)
    df['vol_21d'] = df['daily_return'].rolling(21).std() * np.sqrt(252)
    df['vol_63d'] = df['daily_return'].rolling(63).std() * np.sqrt(252)

    # Momentum
    df['mom_5d'] = df['adjusted_close'].pct_change(5)
    df['mom_21d'] = df['adjusted_close'].pct_change(21)
    df['mom_63d'] = df['adjusted_close'].pct_change(63)

    # Moving averages
    df['sma_21d'] = df['adjusted_close'].rolling(21).mean()
    df['sma_50d'] = df['adjusted_close'].rolling(50).mean()
    df['sma_200d'] = df['adjusted_close'].rolling(200).mean()

    # Price relative to SMAs
    df['price_to_sma21'] = df['adjusted_close'] / df['sma_21d']
    df['price_to_sma50'] = df['adjusted_close'] / df['sma_50d']
    df['price_to_sma200'] = df['adjusted_close'] / df['sma_200d']

    # Trend indicators
    df['sma21_above_sma50'] = (df['sma_21d'] > df['sma_50d']).astype(int)
    df['sma50_above_sma200'] = (df['sma_50d'] > df['sma_200d']).astype(int)

    # Drawdowns
    df['rolling_max_63d'] = df['adjusted_close'].rolling(63).max()
    df['drawdown_63d'] = (df['adjusted_close'] - df['rolling_max_63d']) / df['rolling_max_63d']
    df['rolling_max_252d'] = df['adjusted_close'].rolling(252).max()
    df['drawdown_252d'] = (df['adjusted_close'] - df['rolling_max_252d']) / df['rolling_max_252d']

    # Volume
    df['volume_sma_21d'] = df['volume'].rolling(21).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma_21d']

    # RSI (14-day)
    delta = df['daily_return'].copy()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))

    # Bollinger band width
    df['bb_width'] = (df['adjusted_close'].rolling(21).std() * 2) / df['sma_21d']

    # Volatility regime
    vol_median = df['vol_21d'].expanding().median()
    df['high_vol_regime'] = (df['vol_21d'] > vol_median).astype(int)

    return df

daily_feat = daily.groupby('symbol', group_keys=False).apply(compute_technical_features)
print(f"Technical features computed: {daily_feat.shape}")

Technical features computed: (52486, 34)


## 3. Fundamental Features (Forward-Filled to Daily)

In [4]:
# Select key fundamental columns
income_sub = income[['symbol', 'fiscalDateEnding', 'totalRevenue', 'grossProfit',
                     'operatingIncome', 'netIncome', 'ebitda']].copy()
balance_sub = balance[['symbol', 'fiscalDateEnding', 'totalAssets', 'totalLiabilities',
                       'totalShareholderEquity', 'cashAndCashEquivalentsAtCarryingValue',
                       'shortLongTermDebtTotal']].copy()
cashflow_sub = cashflow[['symbol', 'fiscalDateEnding', 'operatingCashflow',
                         'capitalExpenditures']].copy()

# Filter to existing columns
for df, name in [(income_sub, 'income'), (balance_sub, 'balance'), (cashflow_sub, 'cashflow')]:
    missing = [c for c in df.columns if c not in df.columns]
    if missing:
        print(f'{name}: missing columns {missing}')

# Merge fundamentals
fundamentals = income_sub.merge(balance_sub, on=['symbol', 'fiscalDateEnding'], how='outer')
fundamentals = fundamentals.merge(cashflow_sub, on=['symbol', 'fiscalDateEnding'], how='outer')

# Compute ratios
for col in fundamentals.columns:
    if col not in ['symbol', 'fiscalDateEnding']:
        fundamentals[col] = pd.to_numeric(fundamentals[col], errors='coerce')

fundamentals = fundamentals.sort_values(['symbol', 'fiscalDateEnding'])

def compute_fund_ratios(df):
    df = df.copy().sort_values('fiscalDateEnding')
    df['gross_margin'] = df['grossProfit'] / df['totalRevenue'].replace(0, np.nan)
    df['operating_margin'] = df['operatingIncome'] / df['totalRevenue'].replace(0, np.nan)
    df['net_margin'] = df['netIncome'] / df['totalRevenue'].replace(0, np.nan)
    df['revenue_growth_yoy'] = df['totalRevenue'].pct_change(4)
    df['earnings_growth_yoy'] = df['netIncome'].pct_change(4)
    df['debt_to_equity'] = df['shortLongTermDebtTotal'] / df['totalShareholderEquity'].replace(0, np.nan)
    df['cash_ratio'] = df['cashAndCashEquivalentsAtCarryingValue'] / df['totalLiabilities'].replace(0, np.nan)
    df['roe'] = df['netIncome'] / df['totalShareholderEquity'].replace(0, np.nan)
    df['roa'] = df['netIncome'] / df['totalAssets'].replace(0, np.nan)
    df['freeCashFlow'] = df['operatingCashflow'] - df['capitalExpenditures'].abs()
    return df

fundamentals = fundamentals.groupby('symbol', group_keys=False).apply(compute_fund_ratios)

fund_ratio_cols = ['gross_margin', 'operating_margin', 'net_margin', 'revenue_growth_yoy',
                   'earnings_growth_yoy', 'debt_to_equity', 'cash_ratio', 'roe', 'roa']

print(f"Fundamentals computed: {fundamentals.shape}")

Fundamentals computed: (781, 24)


In [5]:
# Forward-fill fundamentals to daily frequency via merge_asof
merged_parts = []
for sym in UNIVERSE:
    d = daily_feat[daily_feat['symbol'] == sym].sort_values('date')
    f = fundamentals[fundamentals['symbol'] == sym].sort_values('fiscalDateEnding')
    m = pd.merge_asof(
        d, f[['fiscalDateEnding'] + fund_ratio_cols],
        left_on='date', right_on='fiscalDateEnding', direction='backward'
    )
    merged_parts.append(m)

daily_feat = pd.concat(merged_parts, ignore_index=True)
daily_feat = daily_feat.drop(columns=['fiscalDateEnding'], errors='ignore')

print(f"Daily features after fundamentals: {daily_feat.shape}")

Daily features after fundamentals: (52486, 43)


## 4. Valuation Features

In [6]:
# Shares outstanding
shares = overview[['symbol', 'sharesoutstanding']].copy()
shares['sharesoutstanding'] = pd.to_numeric(shares['sharesoutstanding'], errors='coerce')

daily_feat = daily_feat.merge(shares, on='symbol', how='left')

# Market cap and valuation ratios
daily_feat['market_cap'] = daily_feat['adjusted_close'] * daily_feat['sharesoutstanding']

# TTM approximation (4x quarterly)
for col_src, col_ttm in [('netIncome', 'earnings_ttm'), ('totalRevenue', 'revenue_ttm'),
                          ('ebitda', 'ebitda_ttm'), ('freeCashFlow', 'fcf_ttm')]:
    if col_src in daily_feat.columns:
        daily_feat[col_ttm] = daily_feat[col_src] * 4

# Valuation ratios
if 'earnings_ttm' in daily_feat.columns:
    daily_feat['pe_ratio'] = daily_feat['market_cap'] / daily_feat['earnings_ttm'].replace(0, np.nan)
    daily_feat['pe_ratio'] = daily_feat['pe_ratio'].replace([np.inf, -np.inf], np.nan).clip(-100, 500)

if 'revenue_ttm' in daily_feat.columns:
    daily_feat['ps_ratio'] = daily_feat['market_cap'] / daily_feat['revenue_ttm'].replace(0, np.nan)
    daily_feat['ps_ratio'] = daily_feat['ps_ratio'].replace([np.inf, -np.inf], np.nan).clip(-100, 500)

if 'ebitda_ttm' in daily_feat.columns:
    ev = daily_feat['market_cap'] + daily_feat.get('shortLongTermDebtTotal', 0) - daily_feat.get('cashAndCashEquivalentsAtCarryingValue', 0)
    daily_feat['ev_ebitda'] = ev / daily_feat['ebitda_ttm'].replace(0, np.nan)
    daily_feat['ev_ebitda'] = daily_feat['ev_ebitda'].replace([np.inf, -np.inf], np.nan).clip(-100, 500)

if 'fcf_ttm' in daily_feat.columns:
    daily_feat['fcf_yield'] = daily_feat['fcf_ttm'] / daily_feat['market_cap'].replace(0, np.nan)
    daily_feat['fcf_yield'] = daily_feat['fcf_yield'].replace([np.inf, -np.inf], np.nan).clip(-1, 1)

print(f"Valuation features added: {daily_feat.shape}")

Valuation features added: (52486, 45)


## 5. Final Feature Selection and Save

In [7]:
# Drop raw columns, keep only features
drop_cols = ['open', 'high', 'low', 'close', 'adjusted_close', 'volume', 'dividend',
             'split_coefficient', 'sma_21d', 'sma_50d', 'sma_200d',
             'rolling_max_63d', 'rolling_max_252d', 'volume_sma_21d',
             'sharesoutstanding', 'market_cap',
             'totalRevenue', 'grossProfit', 'operatingIncome', 'netIncome', 'ebitda',
             'totalAssets', 'totalLiabilities', 'totalShareholderEquity',
             'cashAndCashEquivalentsAtCarryingValue', 'shortLongTermDebtTotal',
             'operatingCashflow', 'capitalExpenditures', 'freeCashFlow',
             'earnings_ttm', 'revenue_ttm', 'ebitda_ttm', 'fcf_ttm']

daily_feat = daily_feat.drop(columns=[c for c in drop_cols if c in daily_feat.columns], errors='ignore')

# Filter: require at least 252 days of history (1 year warmup)
min_date = '2001-01-01'
daily_feat = daily_feat[daily_feat['date'] >= min_date].copy()

feature_cols = [c for c in daily_feat.columns if c not in ['symbol', 'date']]
print(f"Final daily features: {len(feature_cols)}")
print(f"Features: {feature_cols}")
print(f"Shape: {daily_feat.shape}")
print(f"Date range: {daily_feat['date'].min().date()} to {daily_feat['date'].max().date()}")

# Missing values
missing = daily_feat[feature_cols].isna().mean().sort_values(ascending=False)
print(f"\nMissing values (>5%):")
print(missing[missing > 0.05])

Final daily features: 27
Features: ['daily_return', 'vol_5d', 'vol_21d', 'vol_63d', 'mom_5d', 'mom_21d', 'mom_63d', 'price_to_sma21', 'price_to_sma50', 'price_to_sma200', 'sma21_above_sma50', 'sma50_above_sma200', 'drawdown_63d', 'drawdown_252d', 'volume_ratio', 'rsi_14', 'bb_width', 'high_vol_regime', 'gross_margin', 'operating_margin', 'net_margin', 'revenue_growth_yoy', 'earnings_growth_yoy', 'debt_to_equity', 'cash_ratio', 'roe', 'roa']
Shape: (51226, 29)
Date range: 2001-01-02 to 2025-12-31

Missing values (>5%):
debt_to_equity         0.233260
earnings_growth_yoy    0.159489
revenue_growth_yoy     0.159489
roa                    0.130090
gross_margin           0.130090
roe                    0.130090
cash_ratio             0.130090
net_margin             0.130090
operating_margin       0.130090
dtype: float64


In [8]:
# Save
daily_feat.to_parquet(OUTPUT_DIR / 'daily_features.parquet', index=False)
print(f"Saved: {OUTPUT_DIR / 'daily_features.parquet'}")

# Feature summary
stats = daily_feat[feature_cols].describe().T.round(4)
stats['missing_pct'] = daily_feat[feature_cols].isna().mean().round(4)
stats.to_csv(OUTPUT_DIR / 'feature_summary.csv')
print(f"Saved: {OUTPUT_DIR / 'feature_summary.csv'}")

print(f"\n{'='*60}")
print("DAILY FEATURE ENGINEERING COMPLETE")
print(f"{'='*60}")
print(f"Tickers: {daily_feat['symbol'].nunique()} ({', '.join(UNIVERSE)})")
print(f"Date range: {daily_feat['date'].min().date()} to {daily_feat['date'].max().date()}")
print(f"Total rows: {len(daily_feat):,}")
print(f"Features: {len(feature_cols)}")

Saved: ../data/processed/daily_features.parquet
Saved: ../data/processed/feature_summary.csv

DAILY FEATURE ENGINEERING COMPLETE
Tickers: 10 (AAPL, AMZN, AVGO, GOOG, GOOGL, META, MSFT, NVDA, TSLA, WMT)
Date range: 2001-01-02 to 2025-12-31
Total rows: 51,226
Features: 27
